# Feature Engineering Masterclass
## Learning Objectives
- Create meaningful features from raw data
- Apply encoding, scaling, and transformation techniques
- Use automated feature selection methods
- Handle temporal and text features


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, TargetEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['target'] = housing.target
print(df.head())
print(f'Shape: {df.shape}')

In [ ]:
# Numerical feature transformations
df_fe = df.copy()

# Log transformation for skewed features
df_fe['log_AveOccup'] = np.log1p(df['AveOccup'])
df_fe['log_Population'] = np.log1p(df['Population'])

# Binning
df_fe['income_bin'] = pd.cut(df['MedInc'], bins=5, labels=['very_low','low','mid','high','very_high'])

# Interaction features
df_fe['rooms_per_person'] = df['AveRooms'] / df['AveOccup']
df_fe['bedrooms_per_room'] = df['AveBedrms'] / df['AveRooms']

print('New features created:')
new_cols = ['log_AveOccup','log_Population','income_bin','rooms_per_person','bedrooms_per_room']
print(df_fe[new_cols].describe().round(3))

In [ ]:
# Polynomial features
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

X = df[['MedInc', 'HouseAge', 'AveRooms']].values
y = df['target'].values

for degree in [1, 2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)
    ridge = Ridge(alpha=1.0)
    scores = cross_val_score(ridge, X_poly, y, cv=5, scoring='r2')
    print(f'Degree {degree}: {X_poly.shape[1]:3d} features, R²={scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Feature selection
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.preprocessing import StandardScaler

X_all = df.drop(columns=['target','income_bin'] if 'income_bin' in df.columns else ['target']).select_dtypes(include=[np.number])
y = df['target'].values

# Mutual information
mi_scores = mutual_info_regression(X_all, y, random_state=42)
mi_df = pd.DataFrame({'feature': X_all.columns, 'mi_score': mi_scores}).sort_values('mi_score', ascending=False)
print('Feature Importance (Mutual Information):')
print(mi_df.to_string(index=False))

## Exercises
1. Apply target encoding to a high-cardinality categorical column.
2. Create date-based features (day of week, month, quarter) from a datetime column.
3. Implement a custom `FeatureUnion` pipeline combining numerical and text features.


## Summary
- Feature engineering often has more impact than model selection.
- Log transforms help handle skewed distributions.
- Mutual information is a nonlinear, model-agnostic feature selector.
